In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [2]:
DATA_DIR = Path("../data")
part1 = pd.read_csv("../data/part1_company_year_values.csv")
part2 = pd.read_csv("../data/part2_proxy_disclosure_analysis_normalized.csv")
PART3_OUTPUT_PATH = "../data/part3_authenticity_index.csv"

In [3]:
part1_usable = part1[
    part1["text_extraction_status"] == "success"
].copy()

print(part1_usable.shape)
part1_usable.head()

(226, 15)


,ticker,company_name,sector,year,page_text_clean,changed_from_prior,theme_categories,analyst_notes,about_url,snapshot_url,snapshot_timestamp,status,text_extraction_status,text_length,similarity_to_prior
9,ABBV,AbbVie,Healthcare,2016,Who We Are | AbbVie We Create Medicines and So...,no_prior_available,innovation; customer_focus; trust_ethics; peop...,The extracted page text is 3831 characters lon...,https://www.abbvie.com/who-we-are.html,http://web.archive.org/web/20230216175901/http...,2.023022e+13,found,success,3831,NaN
11,ABBV,AbbVie,Healthcare,2018,Who We Are | AbbVie We Create Medicines and So...,not_changed,innovation; customer_focus; trust_ethics; peop...,The extracted page text is 3831 characters lon...,https://www.abbvie.com/who-we-are.html,http://web.archive.org/web/20230216175901/http...,2.023022e+13,found,success,3831,1.000000
13,ABBV,AbbVie,Healthcare,2020,Who We Are | AbbVie We Create Medicines and So...,not_changed,innovation; customer_focus; trust_ethics; peop...,The extracted page text is 3831 characters lon...,https://www.abbvie.com/who-we-are.html,http://web.archive.org/web/20230216175901/http...,2.023022e+13,found,success,3831,1.000000
15,ABBV,AbbVie,Healthcare,2022,Who We Are | AbbVie We Create Medicines and So...,not_changed,innovation; customer_focus; trust_ethics; peop...,The extracted page text is 3831 characters lon...,https://www.abbvie.com/who-we-are.html,http://web.archive.org/web/20230216175901/http...,2.023022e+13,found,success,3831,1.000000
16,ABBV,AbbVie,Healthcare,2023,Who We Are | AbbVie We Create Medicines and So...,not_changed,innovation; customer_focus; trust_ethics; peop...,The extracted page text is 3916 characters lon...,https://www.abbvie.com/who-we-are.html,http://web.archive.org/web/20230718193918/http...,2.023072e+13,found,success,3916,0.989028


In [16]:
PART1_THEMES = [
    "performance_growth",
    "people_employees",
    "innovation",
    "customer_focus",
    "trust_ethics",
    "sustainability_social_impact",
    "safety_security",
    "diversity_inclusion"
]

for theme in PART1_THEMES:
    part1_usable[f"p1_{theme}"] = part1_usable["theme_categories"].fillna("").apply(
        lambda x: 1 if theme in str(x).split("; ") else 0
    )

part1_usable[["ticker", "year", "theme_categories"] + [f"p1_{t}" for t in PART1_THEMES]].head()

,ticker,year,theme_categories,p1_performance_growth,p1_people_employees,p1_innovation,p1_customer_focus,p1_trust_ethics,p1_sustainability_social_impact,p1_safety_security,p1_diversity_inclusion
9,ABBV,2016,innovation; customer_focus; trust_ethics; peop...,1,1,1,1,1,1,0,1
11,ABBV,2018,innovation; customer_focus; trust_ethics; peop...,1,1,1,1,1,1,0,1
13,ABBV,2020,innovation; customer_focus; trust_ethics; peop...,1,1,1,1,1,1,0,1
15,ABBV,2022,innovation; customer_focus; trust_ethics; peop...,1,1,1,1,1,1,0,1
16,ABBV,2023,innovation; customer_focus; trust_ethics; peop...,1,1,1,1,1,1,1,1


In [17]:
PART2_COLS = [
    "diversity_inclusion_per_1000_words",
    "sustainability_environment_per_1000_words",
    "employees_human_capital_per_1000_words",
    "governance_ethics_per_1000_words",
    "community_social_impact_per_1000_words",
    "risk_security_per_1000_words",
    "forward_looking_count_per_1000_words",
    "risk_language_count_per_1000_words"
]

part2_selected = part2[
    ["ticker", "company_name", "sector", "year"] + PART2_COLS
].copy()

part2_selected.head()

,ticker,company_name,sector,year,diversity_inclusion_per_1000_words,sustainability_environment_per_1000_words,employees_human_capital_per_1000_words,governance_ethics_per_1000_words,community_social_impact_per_1000_words,risk_security_per_1000_words,forward_looking_count_per_1000_words,risk_language_count_per_1000_words
0,AAPL,Apple,Technology,2016.0,2.330851,1.489155,9.690299,1.489155,0.107910,1.877630,9.970864,1.899212
1,AAPL,Apple,Technology,2017.0,4.092812,0.377798,13.254416,2.203822,0.535214,3.022385,5.950320,2.896452
2,AAPL,Apple,Technology,2019.0,3.061940,0.405799,12.505995,2.545468,0.295127,3.984211,5.644299,4.094883
3,AAPL,Apple,Technology,2020.0,2.586870,1.782843,13.004265,2.761658,0.244704,4.299797,6.537090,4.194924
4,AAPL,Apple,Technology,2021.0,3.251583,1.279878,14.562939,3.839635,1.106922,4.669826,6.641530,5.188696


In [18]:
merged = part1_usable.merge(
    part2_selected,
    on=["ticker", "company_name", "sector", "year"],
    how="inner"
)

print(merged.shape)
merged[["ticker", "company_name", "sector", "year"]].head()

(138, 31)


,ticker,company_name,sector,year
0,ABBV,AbbVie,Healthcare,2018
1,ABBV,AbbVie,Healthcare,2020
2,ABBV,AbbVie,Healthcare,2022
3,ABBV,AbbVie,Healthcare,2023
4,ABBV,AbbVie,Healthcare,2024


In [19]:
# Part 1 stated-values vector
merged["p1_people"] = merged["p1_people_employees"]
merged["p1_diversity"] = merged["p1_diversity_inclusion"]
merged["p1_sustainability_impact"] = merged["p1_sustainability_social_impact"]
merged["p1_governance_trust_risk"] = (
    merged["p1_trust_ethics"] + merged["p1_safety_security"]
).clip(upper=1)
merged["p1_growth_strategy"] = (
    merged["p1_performance_growth"] + merged["p1_innovation"]
).clip(upper=1)

P1_VECTOR_COLS = [
    "p1_people",
    "p1_diversity",
    "p1_sustainability_impact",
    "p1_governance_trust_risk",
    "p1_growth_strategy"
]

In [20]:
merged["p2_people"] = merged["employees_human_capital_per_1000_words"]
merged["p2_diversity"] = merged["diversity_inclusion_per_1000_words"]
merged["p2_sustainability_impact"] = (
    merged["sustainability_environment_per_1000_words"] +
    merged["community_social_impact_per_1000_words"]
)
merged["p2_governance_trust_risk"] = (
    merged["governance_ethics_per_1000_words"] +
    merged["risk_security_per_1000_words"]
)
merged["p2_growth_strategy"] = merged["forward_looking_count_per_1000_words"]

P2_VECTOR_RAW_COLS = [
    "p2_people",
    "p2_diversity",
    "p2_sustainability_impact",
    "p2_governance_trust_risk",
    "p2_growth_strategy"
]

In [21]:
scaler = MinMaxScaler()

merged[[col + "_scaled" for col in P2_VECTOR_RAW_COLS]] = scaler.fit_transform(
    merged[P2_VECTOR_RAW_COLS]
)

P2_VECTOR_COLS = [col + "_scaled" for col in P2_VECTOR_RAW_COLS]

merged[P1_VECTOR_COLS + P2_VECTOR_COLS].head()

,p1_people,p1_diversity,p1_sustainability_impact,p1_governance_trust_risk,p1_growth_strategy,p2_people_scaled,p2_diversity_scaled,p2_sustainability_impact_scaled,p2_governance_trust_risk_scaled,p2_growth_strategy_scaled
0,1,1,1,1,1,0.584377,0.184316,0.079812,0.316816,0.700724
1,1,1,1,1,1,0.586200,0.243788,0.114362,0.363260,0.657687
2,1,1,1,1,1,0.561533,0.311947,0.135461,0.382409,0.655595
3,1,1,1,1,1,0.558357,0.394356,0.173458,0.356297,0.590964
4,1,1,1,1,1,0.554673,0.315327,0.171094,0.356065,0.616564


In [24]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def calculate_alignment_score(row):
    """
    Calculate cosine similarity between Part 1 stated-values vector
    and Part 2 disclosure-priorities vector.
    """
    p1_vector = row[P1_VECTOR_COLS].astype(float).values.reshape(1, -1)
    p2_vector = row[P2_VECTOR_COLS].astype(float).values.reshape(1, -1)
    
    # If Part 1 has no identified themes, return NaN
    if p1_vector.sum() == 0:
        return np.nan
    
    score = cosine_similarity(p1_vector, p2_vector)[0][0]
    return score


merged["alignment_score"] = merged.apply(calculate_alignment_score, axis=1)
merged["authenticity_index"] = merged["alignment_score"] * 100

merged[[
    "ticker", 
    "company_name", 
    "sector", 
    "year",
    "alignment_score", 
    "authenticity_index"
]].head()

,ticker,company_name,sector,year,alignment_score,authenticity_index
0,ABBV,AbbVie,Healthcare,2018,0.845922,84.592231
1,ABBV,AbbVie,Healthcare,2020,0.887536,88.753553
2,ABBV,AbbVie,Healthcare,2022,0.912225,91.222459
3,ABBV,AbbVie,Healthcare,2023,0.939778,93.977763
4,ABBV,AbbVie,Healthcare,2024,0.927235,92.723516


In [26]:
base_cols = [
    "ticker",
    "company_name",
    "sector",
    "year",
    "alignment_score",
    "authenticity_index"
]

vector_cols = P1_VECTOR_COLS + P2_VECTOR_COLS

optional_cols = [
    "theme_categories",
    "changed_from_prior",
    "text_extraction_status",
    "text_length",
    "similarity_to_prior",
    "page_text_clean"
]

part3_cols = base_cols + vector_cols + [
    col for col in optional_cols if col in merged.columns
]

part3_df = merged[part3_cols].copy()

part3_df.to_csv(PART3_OUTPUT_PATH, index=False)

print(f"Saved Part 3 output to: {PART3_OUTPUT_PATH}")
print(part3_df.shape)

part3_df.head()

Saved Part 3 output to: ../data/part3_authenticity_index.csv
(138, 22)


,ticker,company_name,sector,year,alignment_score,authenticity_index,p1_people,p1_diversity,p1_sustainability_impact,p1_governance_trust_risk,...,p2_diversity_scaled,p2_sustainability_impact_scaled,p2_governance_trust_risk_scaled,p2_growth_strategy_scaled,theme_categories,changed_from_prior,text_extraction_status,text_length,similarity_to_prior,page_text_clean
0,ABBV,AbbVie,Healthcare,2018,0.845922,84.592231,1,1,1,1,...,0.184316,0.079812,0.316816,0.700724,innovation; customer_focus; trust_ethics; peop...,not_changed,success,3831,1.000000,Who We Are | AbbVie We Create Medicines and So...
1,ABBV,AbbVie,Healthcare,2020,0.887536,88.753553,1,1,1,1,...,0.243788,0.114362,0.363260,0.657687,innovation; customer_focus; trust_ethics; peop...,not_changed,success,3831,1.000000,Who We Are | AbbVie We Create Medicines and So...
2,ABBV,AbbVie,Healthcare,2022,0.912225,91.222459,1,1,1,1,...,0.311947,0.135461,0.382409,0.655595,innovation; customer_focus; trust_ethics; peop...,not_changed,success,3831,1.000000,Who We Are | AbbVie We Create Medicines and So...
3,ABBV,AbbVie,Healthcare,2023,0.939778,93.977763,1,1,1,1,...,0.394356,0.173458,0.356297,0.590964,innovation; customer_focus; trust_ethics; peop...,not_changed,success,3916,0.989028,Who We Are | AbbVie We Create Medicines and So...
4,ABBV,AbbVie,Healthcare,2024,0.927235,92.723516,1,1,1,1,...,0.315327,0.171094,0.356065,0.616564,innovation; customer_focus; trust_ethics; peop...,not_changed,success,3980,0.978217,Who We Are | AbbVie We Create Medicines and So...


In [27]:
part3_df["authenticity_index"].describe()

count    138.000000
mean      82.538777
std       11.578875
min        0.000000
25%       78.379256
50%       84.386978
75%       90.584373
max       98.916537
Name: authenticity_index, dtype: float64

In [28]:
sector_auth_summary = (
    part3_df
    .groupby("sector")["authenticity_index"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

sector_auth_summary

,sector,count,mean,median,min,max
2,Financials,3,90.990968,95.385086,81.972359,95.615459
1,Energy,42,87.894797,87.066341,73.154599,98.916537
3,Healthcare,43,85.746394,86.004023,69.555073,95.303360
0,Consumer Discretionary,38,78.045915,78.837830,58.350928,92.956544
4,Technology,12,64.413102,69.741886,0.000000,80.287974


In [29]:
top_examples = part3_df.sort_values("authenticity_index", ascending=False).head(10)
bottom_examples = part3_df.sort_values("authenticity_index", ascending=True).head(10)

top_examples[[
    "ticker", "company_name", "sector", "year",
    "authenticity_index", "theme_categories"
]]

,ticker,company_name,sector,year,authenticity_index,theme_categories
65,MPC,Marathon Petroleum,Energy,2024,98.916537,innovation; trust_ethics; people_employees; di...
64,MPC,Marathon Petroleum,Energy,2023,97.988243,innovation; trust_ethics; people_employees; di...
134,VLO,Valero Energy,Energy,2024,96.477590,innovation; customer_focus; people_employees; ...
63,MPC,Marathon Petroleum,Energy,2022,95.759573,innovation; trust_ethics; people_employees; di...
26,CVX,Chevron,Energy,2021,95.662182,innovation; customer_focus; trust_ethics; peop...
14,AXP,American Express,Financials,2022,95.615459,innovation; customer_focus; trust_ethics; peop...
15,AXP,American Express,Financials,2023,95.385086,innovation; customer_focus; trust_ethics; peop...
121,TMO,Thermo Fisher Scientific,Healthcare,2024,95.303360,innovation; customer_focus; trust_ethics; peop...
118,TMO,Thermo Fisher Scientific,Healthcare,2020,95.099892,innovation; customer_focus; trust_ethics; peop...
24,CVX,Chevron,Energy,2019,95.042534,innovation; customer_focus; trust_ethics; peop...


In [30]:
bottom_examples[[
    "ticker", "company_name", "sector", "year",
    "authenticity_index", "theme_categories"
]]

,ticker,company_name,sector,year,authenticity_index,theme_categories
80,NVDA,NVIDIA,Technology,2022,0.000000,innovation; customer_focus; trust_ethics; perf...
39,INTC,Intel,Technology,2024,57.439245,innovation; customer_focus; performance_growth
11,AMZN,Amazon,Consumer Discretionary,2022,58.350928,innovation; customer_focus; people_employees; ...
101,SBUX,Starbucks,Consumer Discretionary,2018,58.739294,trust_ethics; sustainability_social_impact; pe...
100,SBUX,Starbucks,Consumer Discretionary,2017,59.019895,trust_ethics; sustainability_social_impact; pe...
110,TGT,Target,Consumer Discretionary,2024,60.896056,people_employees; performance_growth
83,ORCL,Oracle,Technology,2021,60.949905,innovation; customer_focus; people_employees; ...
99,SBUX,Starbucks,Consumer Discretionary,2016,61.734615,trust_ethics; sustainability_social_impact; pe...
38,INTC,Intel,Technology,2019,63.382761,innovation; customer_focus; trust_ethics; perf...
82,NVDA,NVIDIA,Technology,2024,66.839196,innovation; customer_focus; people_employees; ...


In [31]:
part3_df.shape

(138, 22)

In [32]:
part3_df["authenticity_index"].describe()

count    138.000000
mean      82.538777
std       11.578875
min        0.000000
25%       78.379256
50%       84.386978
75%       90.584373
max       98.916537
Name: authenticity_index, dtype: float64

In [33]:
sector_auth_summary = (
    part3_df
    .groupby("sector")["authenticity_index"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

sector_auth_summary

,sector,count,mean,median,min,max
2,Financials,3,90.990968,95.385086,81.972359,95.615459
1,Energy,42,87.894797,87.066341,73.154599,98.916537
3,Healthcare,43,85.746394,86.004023,69.555073,95.303360
0,Consumer Discretionary,38,78.045915,78.837830,58.350928,92.956544
4,Technology,12,64.413102,69.741886,0.000000,80.287974


In [34]:
year_auth_summary = (
    part3_df
    .groupby("year")["authenticity_index"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
    .sort_values("year")
)

year_auth_summary

,year,count,mean,median,min,max
0,2016,5,81.114022,84.440587,61.734615,90.306063
1,2017,4,77.915759,81.032511,59.019895,90.578119
2,2018,9,83.339816,86.875737,58.739294,92.345403
3,2019,15,83.744018,85.533868,63.382761,95.042534
4,2020,17,85.124494,82.397611,77.934769,95.099892
5,2021,21,82.365229,83.368258,60.949905,95.662182
6,2022,20,78.700626,82.752259,0.000000,95.759573
7,2023,25,84.339469,85.884133,69.555073,97.988243
8,2024,22,82.164276,84.427563,57.439245,98.916537


In [35]:
top_examples = (
    part3_df
    .sort_values("authenticity_index", ascending=False)
    .head(10)
)

top_examples[
    [
        "ticker",
        "company_name",
        "sector",
        "year",
        "authenticity_index",
        "theme_categories"
    ]
]

,ticker,company_name,sector,year,authenticity_index,theme_categories
65,MPC,Marathon Petroleum,Energy,2024,98.916537,innovation; trust_ethics; people_employees; di...
64,MPC,Marathon Petroleum,Energy,2023,97.988243,innovation; trust_ethics; people_employees; di...
134,VLO,Valero Energy,Energy,2024,96.477590,innovation; customer_focus; people_employees; ...
63,MPC,Marathon Petroleum,Energy,2022,95.759573,innovation; trust_ethics; people_employees; di...
26,CVX,Chevron,Energy,2021,95.662182,innovation; customer_focus; trust_ethics; peop...
14,AXP,American Express,Financials,2022,95.615459,innovation; customer_focus; trust_ethics; peop...
15,AXP,American Express,Financials,2023,95.385086,innovation; customer_focus; trust_ethics; peop...
121,TMO,Thermo Fisher Scientific,Healthcare,2024,95.303360,innovation; customer_focus; trust_ethics; peop...
118,TMO,Thermo Fisher Scientific,Healthcare,2020,95.099892,innovation; customer_focus; trust_ethics; peop...
24,CVX,Chevron,Energy,2019,95.042534,innovation; customer_focus; trust_ethics; peop...


In [36]:
bottom_examples = (
    part3_df
    .sort_values("authenticity_index", ascending=True)
    .head(10)
)

bottom_examples[
    [
        "ticker",
        "company_name",
        "sector",
        "year",
        "authenticity_index",
        "theme_categories"
    ]
]

,ticker,company_name,sector,year,authenticity_index,theme_categories
80,NVDA,NVIDIA,Technology,2022,0.000000,innovation; customer_focus; trust_ethics; perf...
39,INTC,Intel,Technology,2024,57.439245,innovation; customer_focus; performance_growth
11,AMZN,Amazon,Consumer Discretionary,2022,58.350928,innovation; customer_focus; people_employees; ...
101,SBUX,Starbucks,Consumer Discretionary,2018,58.739294,trust_ethics; sustainability_social_impact; pe...
100,SBUX,Starbucks,Consumer Discretionary,2017,59.019895,trust_ethics; sustainability_social_impact; pe...
110,TGT,Target,Consumer Discretionary,2024,60.896056,people_employees; performance_growth
83,ORCL,Oracle,Technology,2021,60.949905,innovation; customer_focus; people_employees; ...
99,SBUX,Starbucks,Consumer Discretionary,2016,61.734615,trust_ethics; sustainability_social_impact; pe...
38,INTC,Intel,Technology,2019,63.382761,innovation; customer_focus; trust_ethics; perf...
82,NVDA,NVIDIA,Technology,2024,66.839196,innovation; customer_focus; people_employees; ...


In [37]:
top_examples[
    [
        "ticker",
        "company_name",
        "sector",
        "year",
        "authenticity_index"
    ] + P1_VECTOR_COLS + P2_VECTOR_COLS
]

,ticker,company_name,sector,year,authenticity_index,p1_people,p1_diversity,p1_sustainability_impact,p1_governance_trust_risk,p1_growth_strategy,p2_people_scaled,p2_diversity_scaled,p2_sustainability_impact_scaled,p2_governance_trust_risk_scaled,p2_growth_strategy_scaled
65,MPC,Marathon Petroleum,Energy,2024,98.916537,1,1,1,1,1,0.753931,0.476462,0.591515,0.576996,0.597782
64,MPC,Marathon Petroleum,Energy,2023,97.988243,1,1,1,1,1,0.729954,0.372777,0.595716,0.539820,0.586138
134,VLO,Valero Energy,Energy,2024,96.477590,1,0,1,1,1,0.725272,0.359702,0.726038,0.600293,0.691907
63,MPC,Marathon Petroleum,Energy,2022,95.759573,1,1,1,1,1,0.766799,0.267069,0.555030,0.501786,0.594446
26,CVX,Chevron,Energy,2021,95.662182,1,1,1,1,1,0.569401,0.228791,0.715009,0.625532,0.583147
14,AXP,American Express,Financials,2022,95.615459,1,1,0,1,1,0.539724,0.523994,0.225290,0.959418,0.692833
15,AXP,American Express,Financials,2023,95.385086,1,1,0,1,1,0.551916,0.459542,0.155352,0.983188,0.721462
121,TMO,Thermo Fisher Scientific,Healthcare,2024,95.303360,1,0,0,1,1,0.659396,0.259133,0.189141,0.605416,0.924879
118,TMO,Thermo Fisher Scientific,Healthcare,2020,95.099892,1,0,0,1,1,0.708909,0.252092,0.224245,0.545734,0.817805
24,CVX,Chevron,Energy,2019,95.042534,1,1,1,1,1,0.557216,0.190470,0.438261,0.618274,0.589782


In [38]:
bottom_examples[
    [
        "ticker",
        "company_name",
        "sector",
        "year",
        "authenticity_index"
    ] + P1_VECTOR_COLS + P2_VECTOR_COLS
]

,ticker,company_name,sector,year,authenticity_index,p1_people,p1_diversity,p1_sustainability_impact,p1_governance_trust_risk,p1_growth_strategy,p2_people_scaled,p2_diversity_scaled,p2_sustainability_impact_scaled,p2_governance_trust_risk_scaled,p2_growth_strategy_scaled
80,NVDA,NVIDIA,Technology,2022,0.000000,0,0,0,1,1,0.000000,0.000000,0.000000,0.000000,0.000000
39,INTC,Intel,Technology,2024,57.439245,0,0,0,0,1,0.806188,0.497299,0.171887,0.754120,0.858102
11,AMZN,Amazon,Consumer Discretionary,2022,58.350928,1,0,0,0,1,0.871477,1.000000,0.653366,0.454991,0.460525
101,SBUX,Starbucks,Consumer Discretionary,2018,58.739294,0,0,1,1,1,0.596128,0.505942,0.092797,0.426808,0.549329
100,SBUX,Starbucks,Consumer Discretionary,2017,59.019895,0,0,1,1,1,0.466780,0.220311,0.010369,0.351211,0.392173
110,TGT,Target,Consumer Discretionary,2024,60.896056,1,0,0,0,1,0.530033,0.353925,0.374627,0.902436,0.599550
83,ORCL,Oracle,Technology,2021,60.949905,1,0,1,0,1,0.664700,0.799902,0.047541,0.455021,0.692421
99,SBUX,Starbucks,Consumer Discretionary,2016,61.734615,0,0,1,1,1,0.629041,0.293784,0.036044,0.411197,0.685602
38,INTC,Intel,Technology,2019,63.382761,0,0,0,1,1,0.667522,0.587301,0.193103,0.441771,0.623032
82,NVDA,NVIDIA,Technology,2024,66.839196,1,0,1,0,1,0.697272,0.594180,0.168861,0.670070,0.675247
